<a href="https://colab.research.google.com/github/Sakni-Tasnim/clinic-rag-assistant/blob/main/clinic_rag_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-openrouter \
    langchain-huggingface \
    langchain-chroma \
    langchain-text-splitters \
    chromadb \
    pypdf \
    sentence-transformers

In [ ]:

import os

from google.colab import files

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_huggingface import (
    HuggingFaceEmbeddings
)

from langchain_chroma import Chroma

from langchain_openrouter import ChatOpenRouter

from langchain_core.prompts import (
    ChatPromptTemplate
)

/tmp/ipykernel_24213/1535480076.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [ ]:
from google.colab import userdata

api_key = userdata.get('OpenRouterKey')

In [ ]:
import glob

for f in glob.glob("/content/examens*") + glob.glob("/content/horaires*") + \
          glob.glob("/content/presentation_clinique*") + glob.glob("/content/reglement*") + \
          glob.glob("/content/services_medicaux*"):
    os.remove(f)


print("Sélectionnez vos documents PDF / TXT")

uploaded= files.upload()

import shutil
if os.path.exists("documents"):
    shutil.rmtree("documents")
os.makedirs("documents")

os.makedirs("documents", exist_ok = True)

for filename, content in uploaded.items():
  path = os.path.join(
      "documents",
      filename
  )

  with open(path, "wb") as f:
    f.write(content)
  print('Ajouté : ', filename)

Sélectionnez vos documents PDF / TXT


Saving examens.pdf to examens.pdf
Saving horaires.pdf to horaires.pdf
Saving presentation_clinique.pdf to presentation_clinique.pdf
Saving reglement.txt to reglement.txt
Saving services_medicaux.pdf to services_medicaux.pdf
Ajouté :  examens.pdf
Ajouté :  horaires.pdf
Ajouté :  presentation_clinique.pdf
Ajouté :  reglement.txt
Ajouté :  services_medicaux.pdf


In [ ]:
documents = []

for filename in os.listdir("documents"):
  path = os.path.join(
      "documents",
       filename
  )

  if filename.lower().endswith(".pdf"):
    loader = PyPDFLoader(path)
    docs = loader.load()
    documents.extend(docs)
    print(
        f"{filename} → {len(docs)} pages"
    )

  elif filename.lower().endswith(".txt"):
    loader = TextLoader(
        path,
        encoding = "utf-8"
    )
    docs = loader.load()
    documents.extend(docs)
    print(
        f"{filename} => {len(docs)} document"
    )

print("\n Numbre totale de documents/pages: ", len(documents))

services_medicaux.pdf → 6 pages
presentation_clinique.pdf → 3 pages
examens.pdf → 7 pages
horaires.pdf → 4 pages
reglement.txt => 1 document

 Numbre totale de documents/pages:  21


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 150
)

chunks = text_splitter.split_documents(documents)
print(
    "\n✂️ Nombre de chunks: ",
    len(chunks)

)


✂️ Nombre de chunks:  69


In [ ]:
print(
    "\n Chargement du modèle d'embedding..."
)

embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    "\n Chargement terminé !"
)


 Chargement du modèle d'embedding...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


 Chargement terminé !


In [ ]:
print(
    "\n Création de la Vector Database..."
)

vector_store = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    collection_name = "Assistant_medical"
)

print(
    "chromaDB créé"
)




 Création de la Vector Database...
chromaDB créé


In [ ]:
retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs = {
        "k": 5
    }
)
print(
    "Retriever configuré"
)

Retriever configuré


In [ ]:
llm = ChatOpenRouter(
    model = "nvidia/nemotron-3.5-lightning:free",
    api_key = api_key,
    temperature = 0.2
)

print(
    "OpenRouter connecté"
)

OpenRouter connecté


In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """Tu es un assistant virtuel de la Clinique Sainte-Marie.

RÈGLES STRICTES :
1. Réponds UNIQUEMENT à partir du contexte fourni ci-dessous
2. Si la réponse est répartie sur plusieurs extraits du contexte, combine-les
3. Si le contexte ne contient PAS la réponse, dis "Je ne trouve pas cette information dans les documents disponibles"
4. Ne donne JAMAIS de diagnostic médical
5. Réponds en français, de façon claire et concise (1-2 phrases)

CONTEXTE :
{context}"""),
    ("human", "{question}")
])


In [ ]:


print("\n")
print("=" * 60)
print("🤖 CHAT WITH YOUR DOCUMENTS")
print("=" * 60)

print(
    "\nPosez vos questions sur les documents."
)

print(
    "Tapez 'exit' pour quitter.\n"
)
while True:

    # --------------------------------------------------------
    # A. QUESTION UTILISATEUR
    # --------------------------------------------------------

    question = input("👤 Vous : ")

    if question.lower() == "exit":

        print(
            "\n👋 Fin de la conversation."
        )

        break


    # --------------------------------------------------------
    # B. RETRIEVAL
    # --------------------------------------------------------
    #
    # IMPORTANT :
    #
    # Ici LangChain fait automatiquement :
    #
    # question
    #     ↓
    # embedding(question)
    #     ↓
    # recherche vectorielle
    #     ↓
    # Top-K chunks
    #
    # Nous n'avons PAS besoin de faire
    # embeddings.embed_query(question)
    # nous-mêmes.
    #
    # --------------------------------------------------------

    retrieved_docs = retriever.invoke(
        question
    )


    # --------------------------------------------------------
    # C. AFFICHER LES DOCUMENTS RETROUVÉS
    # --------------------------------------------------------

    print(
        "\n🔎 ===== DOCUMENTS RETRIEVED ====="
    )

    for i, doc in enumerate(
        retrieved_docs,
        start=1
    ):

        print(
            f"\n--- Document {i} ---"
        )

        print(
            "Source :",
            doc.metadata.get(
                "source",
                "Inconnue"
            )
        )

        if "page" in doc.metadata:

            print(
                "Page :",
                doc.metadata["page"] + 1
            )

        print(
            "Contenu :",
            doc.page_content[:500]
        )


    # --------------------------------------------------------
    # D. CONSTRUCTION DU CONTEXTE
    # --------------------------------------------------------

    context = "\n\n".join(

        doc.page_content

        for doc in retrieved_docs
    )


    # --------------------------------------------------------
    # E. CONSTRUCTION DU PROMPT
    # --------------------------------------------------------

    messages = prompt.invoke({

        "context": context,

        "question": question
    })


    # --------------------------------------------------------
    # F. GENERATION AVEC LE LLM
    # --------------------------------------------------------

    response = llm.invoke(
        messages
    )


    # --------------------------------------------------------
    # G. AFFICHER LA REPONSE
    # --------------------------------------------------------

    print(
        "\n🤖 Assistant :"
    )

    print(
        response.content
    )


    # --------------------------------------------------------
    # H. AFFICHER LES SOURCES
    # --------------------------------------------------------

    print(
        "\n📚 ===== SOURCES ====="
    )

    for doc in retrieved_docs:

        source = doc.metadata.get(
            "source",
            "Inconnue"
        )

        page = doc.metadata.get(
            "page"
        )

        if page is not None:

            print(
                f"• {source} — page {page + 1}"
            )

        else:

            print(
                f"• {source}"
            )

    print(
        "\n" + "=" * 60
    )





🤖 CHAT WITH YOUR DOCUMENTS

Posez vos questions sur les documents.
Tapez 'exit' pour quitter.

👤 Vous : Quels sont les horaires des urgences ?

🔎 ===== DOCUMENTS RETRIEVED =====

--- Document 1 ---
Source : documents/reglement.txt
Contenu : **Lettre de sortie** : remise au patient ou envoyée au médecin traitant sous 48h

**Suite de soins** : orientation vers des structures adaptées (SSR, HAD, etc.) si nécessaire

---

TITRE VII - URGENCES MÉDICALES

Article 21 - ORGANISATION DES URGENCES
Le service des urgences est ouvert 24h/24, 7j/7.

**Accès** :
- Entrée dédiée au rez-de-chaussée (avenue des Lilas)
- Stationnement prioritaire pour véhicules d'urgence
- Accès direct pour les ambulances

**Prise en charge** :
- Triage par une infi

--- Document 2 ---
Source : documents/services_medicaux.pdf
Page : 6
Contenu : Le service des urgences est ouvert 24h/24, 7j/7, toute l'année. Il est équipé pour accueillir toutes les 
urgences médicales et chirurgicales.
Capacité : 15 boxes d'examen, 3 s

In [ ]:
!pip install -qU gradio --upgrade

import gradio as gr
print(gr.__version__)

def repondre(question, historique):
    # --- Retrieval (same as your console version) ---
    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # --- Génération ---
    messages = prompt.invoke({"context": context, "question": question})
    response = llm.invoke(messages)

    # --- Construction des sources (dédupliquées) ---
    sources_vues = set()
    sources_texte = []

    for doc in retrieved_docs:
        source = doc.metadata.get("source", "Inconnue")
        page = doc.metadata.get("page")

        if page is not None:
            cle = f"{source} — page {page + 1}"
        else:
            cle = source

        if cle not in sources_vues:
            sources_vues.add(cle)
            sources_texte.append(f"📄 {cle}")

    sources_bloc = "\n".join(sources_texte)

    reponse_finale = f"{response.content}\n\n---\n**Sources :**\n{sources_bloc}"

    return reponse_finale


demo = gr.ChatInterface(
    fn=repondre,
    title="🏥 Assistant de la Clinique Sainte-Marie",
    description="Posez une question sur les services, horaires, examens ou le règlement de la clinique.",

    examples=[
        "Quels sont les horaires des urgences ?",
        "Quels sont les horaires de radiologie ?",
        "Que dois-je faire avant une prise de sang à jeun ?",
        "Quel est le tarif d'une consultation ?",
    ],
)

demo.launch(
    share=True,
    debug=True,
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="sky"),
)

6.26.0
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://45df6e1dc2700c200a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
